# 第 11 章 · warp shuffle 与原子操作：不经过 shared memory 交换数据

本章回答：同一个 warp 的线程能不能直接看对方寄存器里的值？`__shfl_down_sync` 怎么用 5 步把 32 个数加起来？归约怎么分层（线程 → warp → block → 全局）？原子操作是什么，什么时候用、什么时候会很慢？

配套讲义：仓库里 `cuda-programming/notes/lessons/11-warp-shuffle-atomics.md`（讲解、动画、练习都在讲义里）。对应原文：Oxford lec4 第 2–6 页（`materials/text/oxford-lec4.txt` 第 13–161 行）、第 17–19 页（第 368–412 行，atomicAdd 与 atomicCAS 锁）；PG §5.4.6.5 Warp Shuffle Functions（`text/cuda-programming-guide.txt` 第 34656–34970 行）、§5.4.6.4 Warp Reduce Functions（第 34620–34655 行）、§2.3.5 Atomics（第 4274–4416 行）；Quack《Getting Memory-bound Kernels to Speed-of-Light》（在线）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 `__shfl_down_sync` | lane t 拿 lane t + d | `warp_sum_down(v)`（8 个 lane，d = 4, 2, 1） | 8 10 6 7 → 14 17 → 31 |
| 第二步 `__shfl_xor_sync` | lane t 拿 lane t XOR m | `warp_sum_xor(v)` | 8 个 lane 全是 31 |
| 第二步 32 个 lane | $\log_2 32$ 步 | `warp_sum_down(w)` | 5 步，lane 0 = np.sum |
| 第四步 原子加 | 读-改-写一次做完 | `counter(1000, atomic=True)` | 1000 |
| 第四步 普通写 | 读和写被别人插进来 | `counter(1000, atomic=False)` | 远小于 1000 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `counter`**（来自 `ch11_shuffle_atomics.py`）：n_threads 个线程各做一次 total += 1。

In [ ]:
def counter(n_threads: int, atomic: bool, seed: int = 0) -> int:
    """n_threads 个线程各做一次 total += 1。

    普通写：每个线程分"读旧值"和"写旧值 + 1"两步，各线程的这两步被随机交错 → 有的加法被覆盖。
    原子加：读-改-写不可打断，一个一个排队。
    """
    total = 0
    if atomic:
        for _ in range(n_threads):
            total += 1
        return total
    rng = np.random.default_rng(seed)
    order = rng.permutation(np.repeat(np.arange(n_threads), 2))  # 每个线程号出现两次：第一次是读，第二次是写
    seen: dict[int, int] = {}
    for t in order:
        if t not in seen:
            seen[t] = total          # 读旧值（放进寄存器）
        else:
            total = seen[t] + 1      # 写回旧值 + 1：期间别人写的都被覆盖
    return total

**函数 `shfl_down`**（来自 `ch11_shuffle_atomics.py`）：lane t 拿 lane t + d 的值；超出 warp 的 lane 拿回自己的值（CUDA 的规定）。

In [ ]:
def shfl_down(v: np.ndarray, d: int) -> np.ndarray:
    """lane t 拿 lane t + d 的值；超出 warp 的 lane 拿回自己的值（CUDA 的规定）。"""
    width = len(v)
    src = np.arange(width) + d
    src[src >= width] = np.arange(width)[src >= width]
    return v[src]

**函数 `warp_sum_down`**（来自 `ch11_shuffle_atomics.py`）：for d = width/2 … 1：v += shfl_down(v, d)。返回每一步之后所有 lane 的值。

In [ ]:
def warp_sum_down(v: np.ndarray) -> list[np.ndarray]:
    """for d = width/2 … 1：v += shfl_down(v, d)。返回每一步之后所有 lane 的值。"""
    v = v.copy()
    steps = []
    d = len(v) // 2
    while d > 0:
        v = v + shfl_down(v, d)
        steps.append(v.copy())
        d //= 2
    return steps

**函数 `shfl_xor`**（来自 `ch11_shuffle_atomics.py`）：lane t 拿 lane (t XOR m) 的值。

In [ ]:
def shfl_xor(v: np.ndarray, m: int) -> np.ndarray:
    """lane t 拿 lane (t XOR m) 的值。"""
    return v[np.arange(len(v)) ^ m]

**函数 `warp_sum_xor`**（来自 `ch11_shuffle_atomics.py`）：蝴蝶归约：for m = 1, 2, 4…：v += shfl_xor(v, m)；最后每个 lane 都是总和。

In [ ]:
def warp_sum_xor(v: np.ndarray) -> np.ndarray:
    """蝴蝶归约：for m = 1, 2, 4…：v += shfl_xor(v, m)；最后每个 lane 都是总和。"""
    v = v.copy()
    m = 1
    while m < len(v):
        v = v + shfl_xor(v, m)
        m *= 2
    return v

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
v = np.array([3, 1, 4, 1, 5, 9, 2, 6])

In [ ]:
print("\n第二步 __shfl_down_sync，8 个 lane，d = 4, 2, 1")
for d, row in zip((4, 2, 1), warp_sum_down(v)):
    print(f"  d = {d}: lane 0–{d - 1} = {row[:d]}（其余 lane 也在算，但没人用）")
print("  lane 0 =", warp_sum_down(v)[-1][0])

In [ ]:
print("\n第二步 __shfl_xor_sync：每个 lane 都拿到总和")
print("  ", warp_sum_xor(v))

In [ ]:
print("\n第二步 32 个 lane：5 步")
rng = np.random.default_rng(1)
w = rng.integers(0, 10, 32)
print(f"  步数 = {len(warp_sum_down(w))}，lane 0 = {warp_sum_down(w)[-1][0]}，np.sum = {w.sum()}")

In [ ]:
print("\n第四步 1000 个线程各加 1")
print("  atomicAdd：", counter(1000, atomic=True))
print("  total += 1（随机交错）：", counter(1000, atomic=False), "——丢了很多次加法，每次跑结果都可能不同")

## 3. 代码里才看得到的

1. **所有 lane 都在算**：`warp_sum_down` 每一步对整个数组做加法，lane 4 以上也算出了值（打印出来不是讲义表里的"—"），只是没人用——和讲义说的"它们在同一条指令里一起执行"一致。
2. **超出范围的 lane 拿回自己的值**：这是 CUDA 对 `__shfl_down_sync` 的规定；所以最后几个 lane 的"和"是错的，只有 lane 0 是对的。
3. **`counter(atomic=False)` 的交错是随机打乱的**：模拟里每个线程的"读"和"写"之间可以插进任意多别人的操作，所以丢得特别多；真实 GPU 上丢多少取决于调度，结果每次都可能不同。

## 4. 自己改一改

- 把 `warp_sum_down` 的输入换成 16 个 lane 的数组，看要几步、lane 0 是不是总和。
- 把 `counter` 的线程数改成 10、100，看普通写丢了多少。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 同一个 warp 直接递纸条，不过讲台、不用全班等；
> down 往大号拿，up 往小号拿，xor 找二进制伙伴，sync 指名道姓；
> 32 个数 5 步加完：16、8、4、2、1；
> 归约四层：线程里多加，warp 里 shuffle，block 里 shared，全局原子加；
> 原子不丢账但要排队：一个 block 一次可以，一个线程一次太慢。